In [8]:
from langchain.agents import AgentState

class ColorContext(AgentState):
  fav_color: str
  least_fav_color: str

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import HumanMessage, ToolMessage

model = init_chat_model(
  model="openai/gpt-oss-120b",
  model_provider="groq",

)

@tool(description="update color's of user in state, once they revealed it")
def update_users_color(fav_color: str, least_fav_color: str, runtime: ToolRuntime) -> Command:
  return Command(update={
    "fav_color":fav_color,
    "least_fav_color": least_fav_color,
    "messages":[ToolMessage(content="successfully updated colors", tool_call_id=runtime.tool_call_id)]
  })

In [ ]:
from pprint import pprint
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(
  model=model,
  tools=[update_users_color],
  checkpointer=InMemorySaver()
)

while(True):
  userInput = input("Ask anything: ")
  if userInput.lower() == "exit":
    break

  res = agent.invoke(
    {"messages": [HumanMessage(content=userInput)]},
    {"configurable": {"thread_id":"01"}}
  )
  pprint(res)
